Оператор группировки спутников внедряет автоматизированную систему предиктивной диагностики. Система по данным телеметрии оценивает риск серьёзного отказа критических подсистем спутника.
В таблице space_risk_model.csv для каждого орбитального витка приведены агрегированные признаки состояния:

    параметры энергосистемы (напряжение и ток батарей);
    температуры ключевых модулей;
    нагрузка на систему ориентации;
    данные о манёврах.

Для каждого витка известно, произошёл ли в ближайшие 12 часов серьёзный отказ (переход в Safe Mode, потеря ориентации, сбой BMS). Такие события редки, но крайне нежелательны: они ведут к высоким затратам и рискам.

На этих данных обучены 10 моделей. Каждая выдаёт оценку риска - число от 0 до 1, которое можно интерпретировать как вероятность отказа в ближайшие 12 часов.
Оператор использует пороговое правило:

    если прогноз модели > порога t, виток считается опасным - назначается проверка или манёвр;
    если прогноз ≤ t, виток считается безопасным - полёт идёт по плану.

Далее следует описание данных.
Каждая строка файла соответствует одному орбитальному витку (наблюдению) для некоторого спутника. Данные отсортированы по времени внутри каждого спутника. Всего наблюдений 180300. В таблице представлены следующие столбцы:

    satellite_id - идентификатор спутника в группировке
    orbit_number - номер витка
    orbit_type - тип орбиты (фиксирован для каждого спутника):
        - 0 - низкая околоземная (LEO, высота 400–600 км, период ~90 мин);
        - 1 - средневысотная (MEO, ~20 000 км, период ~12 ч);
        - 2 - геостационарная (GEO, 35 786 км, период 24 ч).
    failure_in_12h - целевая метка для бинарной классификации:
        - 0 - отказа не произошло в ближайшие 12 часов;
        - 1 - серьёзный отказ (переход в Safe Mode, потеря ориентации, сбой BMS) произошёл в течение ближайших 12 часов.

Доля отказов в датасете составляет около 2%. Это значение искусственно увеличено по сравнению с реальной эксплуатационной статистикой (< 0.1% витков) для обеспечения возможности обучения моделей.

    solar_panel_temp_mean - средняя температура солнечных панелей за виток
    battery_temp_mean - средняя температура аккумуляторов за виток
    battery_voltage_mean - среднее напряжение основной батареи за виток
    battery_current_mean - средний ток энергосистемы за виток
    battery_voltage_std - стандартное отклонение напряжения за виток
    battery_current_std - стандартное отклонение тока за виток
    computer_temp_mean - средняя температура бортового компьютера
    transmitter_temp_mean - средняя температура СВЧ-передатчика
    reaction_wheel_current_mean - средний ток маховиков системы ориентации за виток
    reaction_wheel_current_std - стандартное отклонение тока маховиков за виток
    attitude_error_mean - средняя ошибка ориентации (отклонение от целевого положения)
    thruster_firings_last_24h - количество включений (импульсов) двигателей за последние 24 часа

Прогнозы моделей
Для каждой из 10 моделей предиктивной диагностики в таблице приведён столбец с оценкой риска серьёзного отказа (вероятность от 0 до 1):

    model_1_pred - прогноз первой модели, число 0.0…1.0.
    model_2_pred - прогноз второй модели, число 0.0…1.0.
    model_3_pred - прогноз третьей модели, число 0.0…1.0.
    model_4_pred - прогноз четвёртой модели, число 0.0…1.0.
    model_5_pred - прогноз пятой модели, число 0.0…1.0.
    model_6_pred - прогноз шестой модели, число 0.0…1.0.
    model_7_pred - прогноз седьмой модели, число 0.0…1.0.
    model_8_pred - прогноз восьмой модели, число 0.0…1.0.
    model_9_pred - прогноз девятой модели, число 0.0…1.0.
    model_10_pred - прогноз десятой модели, число 0.0…1.0.

Каждый из этих столбцов содержит оценку вероятности того, что для данного витка failure_in_12h=1. Модели могут вести себя по‑разному (агрессивные и консервативные стратегии), что позволяет анализировать их качество, пороги, потери и ансамбли в рамках подзадач.

In [34]:
import pandas as pd
import numpy as np

In [35]:
df = pd.read_csv("data/space_risk_model.csv")

df.info()

ans = pd.read_csv("data/sample_submission_space_risk_model.csv")

<class 'pandas.DataFrame'>
RangeIndex: 180300 entries, 0 to 180299
Data columns (total 26 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   satellite_id                 180300 non-null  int64  
 1   orbit_number                 180300 non-null  int64  
 2   orbit_type                   180300 non-null  int64  
 3   failure_in_12h               180300 non-null  int64  
 4   solar_panel_temp_mean        180300 non-null  float64
 5   battery_temp_mean            180300 non-null  float64
 6   battery_voltage_mean         180300 non-null  float64
 7   battery_current_mean         180300 non-null  float64
 8   battery_voltage_std          180300 non-null  float64
 9   battery_current_std          180300 non-null  float64
 10  computer_temp_mean           180300 non-null  float64
 11  transmitter_temp_mean        180300 non-null  float64
 12  reaction_wheel_current_mean  180300 non-null  float64
 13  reaction_w

In [36]:
ans = ans.astype(str)

1. Для каждого спутника вычислите среднюю температуру солнечных панелей solar_panel_temp_mean по всем его виткам. Затем для каждого типа орбиты определите спутник с максимальной средней температурой солнечных панелей. В качестве ответа укажите три идентификатора спутников - по одному для LEO, MEO и GEO - в этом порядке. 

In [55]:
means = df.groupby(
    ["orbit_type", "satellite_id"]
)["solar_panel_temp_mean"].mean()

ids = [int(means.loc[o].idxmax()) for o in (0, 1, 2)]
ans.loc[0, ["answer_1", "answer_2", "answer_3"]] = list(map(str, ids))

ans.iloc[0]

question_id     1
answer_1       15
answer_2        3
answer_3        1
Name: 0, dtype: str

2. Определите пять лучших из десяти моделей по качеству ранжирования прогнозируемого риска отказа. Для сравнения выберите наиболее релевантную классическую порогонезависимую метрику качества ранжирования. Метрики, требующие выбора фиксированного порога, и метрики калибровки вероятностей не рассматриваются. При одинаковых значениях прогноза наблюдения сохраняют исходный порядок, заданный таблицей: сначала по satellite_id, затем по orbit_number. Метрика рассчитывается по полученному полному ранжированию, без объединения наблюдений с одинаковыми прогнозами в одну точку. В качестве первого ответа укажите через запятую номера пяти лучших моделей в порядке убывания значения метрики (при равных значениях метрики модель с меньшим номером ставится выше), в качестве второго — через запятую значения выбранной метрики для этих моделей в том же порядке, каждое округлённое до 2 знаков. 

In [38]:
sorted_idx = np.argsort(
    -df[[f"model_{i}_pred" for i in range(1, 11)]].to_numpy(),
    axis=0,
    kind="stable",
).T

n = len(df)
precisions = np.zeros(10)

target = df["failure_in_12h"].to_numpy()

for i in range(10):
    curr_target = target[sorted_idx[i]]
    tp = 0
    for j in range(0, n):
        tp += curr_target[j] == 1
        if curr_target[j]: precisions[i] += (tp)/(j+1)
    precisions[i] /= target.sum()

best_idx = np.argsort(-precisions, kind="stable")[:5]

def commastring(a):
    return ", ".join(a.astype(str))

ans.loc[1, ["answer_1", "answer_2", "answer_3"]] = [
    commastring(
        best_idx+1
    ),
    commastring(
        np.round(precisions[best_idx], 2)
    ),
    pd.NA
]
ans.iloc[1]

question_id                               2
answer_1                     10, 9, 7, 5, 4
answer_2       0.92, 0.48, 0.31, 0.27, 0.16
answer_3                                NaN
Name: 1, dtype: str

3. Добавьте признак скорости нагрева как разность температур за 3 витка:
heating_rate(k) = solar_panel_temp_mean(k) - solar_panel_temp_mean(k-3)
где k - номер текущего орбитального витка для данного спутника.
При этом: расчет heating_rate выполняется отдельно для каждого спутника (satellite_id); для первых 3 витков каждого спутника (где k-3 < 0) значение heating_rate не определено и исключается из всех дальнейших расчетов; при расчете средних значений используется абсолютная величина скорости нагрева: |heating_rate|.
Рассчитайте среднюю скорость нагрева (в °C за 3 витка):
в витках без отказа - для всех витков, где failure_in_12h = 0 (отказ не произошел в ближайшие 12 часов);
за 3 витка до отказа - для каждого витка, где failure_in_12h = 1 (произошел серьезный отказ), берется значение heating_rate на витке с номером k-3 (ровно за 3 витка до отказа). Если для такого витка значение heating_rate не определено (первые 3 витка спутника), этот случай исключается из расчета.
Во сколько раз средняя абсолютная скорость нагрева за 3 витка до отказа выше, чем в витках без отказа? Ответ округлить до целых. 

In [39]:
df["heating_rate"] = (
    df.groupby("satellite_id")["solar_panel_temp_mean"].diff(3)
)

rate_before_failure = df.groupby("satellite_id")["heating_rate"].shift(3)

mean_no_failure = df.loc[
    df["failure_in_12h"] == 0,
    "heating_rate"
].abs().mean()

mean_before_failure = rate_before_failure[
    df["failure_in_12h"] == 1
].abs().mean()

result = round(mean_before_failure / mean_no_failure)

ans.loc[2, ["answer_1", "answer_2", "answer_3"]] = [
    str(result),
    pd.NA,
    pd.NA
]
ans.iloc[2]

question_id      3
answer_1         7
answer_2       NaN
answer_3       NaN
Name: 2, dtype: str

4. Для каждой модели определите минимальное значение её прогноза среди витков, для которых failure_in_12h = 1. Обозначьте найденное значение через t. Для классификации используйте правило: виток считается опасным, если prediction >= t; в противном случае виток считается безопасным (в этом вопросе, в отличие от общего правила «> t» из вступления, используется нестрогое неравенство). При таком выборе порога Recall каждой модели равен 1.0. Для каждой модели рассчитайте долю ложных тревог FPR. Расчёт выполняется по всем виткам без разделения по спутникам. Среди всех моделей выберите модель с минимальным FPR. Укажите номер модели и значение FPR, округлённое до сотых. 

In [40]:
t = df[df["failure_in_12h"] == 1][[f"model_{i}_pred" for i in range(1, 11)]].min(axis=0)
fpr = np.zeros(10)
target = df["failure_in_12h"].to_numpy()
for i in range(10):
    preds = df[f"model_{i+1}_pred"] >= t[f"model_{i+1}_pred"]
    fpr[i] = ((preds == 1) & (target == 0)).sum() / (target == 0).sum()

best_idx = fpr.argmin()

ans.loc[3, ["answer_1", "answer_2", "answer_3"]] = [
    str(best_idx+1),
    str(round(fpr[best_idx], 2)),
    pd.NA
]
ans.iloc[3]

question_id       4
answer_1         10
answer_2       0.29
answer_3        NaN
Name: 3, dtype: str

5. 
Вычислите кросс-корреляцию со сдвигом (lag) между двумя временными рядами: solar_panel_temp_mean - температура солнечных панелей; battery_temp_mean - температура батарей. Расчёт необходимо выполнить отдельно для каждого спутника, для каждого спутника вычислите коэффициент корреляции Пирсона для всех сдвигов lag от -5 до +5 витков включительно.
- При lag>0: r(lag) = corr(солнечные_панели(t), батареи(t+lag)) - батареи запаздывают на lag витков.
- При lag<0: r(lag) = corr(солнечные_панели(t), батареи(t+lag)) - батареи опережают на |lag| витков (так как t + lag < t).
- При lag=0: r(0) = corr(солнечные_панели(t), батареи(t))

Далее необходимо провести усреднение по спутникам.
При расчёте корреляции для каждого спутника используются только доступные пары наблюдений. Например, при lag = 1 используются пары (t, t+1) для всех доступных t.
При усреднении используйте простое среднее: для каждого значения lag сначала вычислите корреляцию по каждому спутнику, затем усредните полученные значения r(lag) по всем спутникам в выборке. При каком сдвиге (lag) средняя корреляция r̄(lag) максимальна? Укажите lag и значение корреляции, округлённое до сотых. 

In [41]:
corr_satellite_lag = np.zeros((30, 11))

for satellite in range(30):
    df_satellite = df[df["satellite_id"] == satellite+1]
    for lag in range(-5, 6):
        corr_satellite_lag[satellite, lag + 5] = (
            df_satellite["solar_panel_temp_mean"].corr(
                df_satellite["battery_temp_mean"].shift(-lag)
            )
        )

corr_lag = corr_satellite_lag.mean(axis=0)
best_idx = corr_lag.argmax()

ans.loc[4, ["answer_1", "answer_2", "answer_3"]] = [
    str(best_idx-5),
    str(round(corr_lag[best_idx], 2)),
    pd.NA
]
ans.iloc[4]

question_id       5
answer_1          1
answer_2       0.78
answer_3        NaN
Name: 4, dtype: str

6. 
Проверим гипотезу о том, что деградация солнечных панелей может быть связана с ростом температуры панелей (solar_panel_temp_mean), ростом тока батарей (battery_current_mean) и снижением напряжения (battery_voltage_mean). Частная корреляция позволяет измерить связь между двумя переменными, очищенную от влияния третьей переменной. Рассчитайте частные коэффициенты корреляции для следующих пар признаков:

- X=solar_panel_temp_mean,
- Y=battery_current_mean, 
- с контролируемой переменной Z=battery_voltage_mean

и

- X=battery_current_mean, 
- Y=battery_voltage_mean, 
- с контролируемой переменной Z=solar_panel_temp_mean.

Расчёт необходимо выполнить на всех витках без разделения по спутникам по всем доступным наблюдениям.
Какая из двух частных корреляций сильнее по модулю? Укажите это значение по модулю, округлённое до сотых. 

In [42]:
import pingouin as pg

data1 = pd.DataFrame({
    'X': df["solar_panel_temp_mean"],
    'Y': df["battery_current_mean"],    
    'Z': df["battery_voltage_mean"]
})

data2 = pd.DataFrame({
    'X': df["battery_current_mean"],
    'Y': df["battery_voltage_mean"],
    'Z': df["solar_panel_temp_mean"],
})

ans.loc[5, ["answer_1", "answer_2", "answer_3"]] = [
    str(
        round(
            max(
                abs(data1.pcorr().loc["X", 'Y']),
                abs(data2.pcorr().loc["X", 'Y'])
            ),
            2
        )
    ),
    pd.NA,
    pd.NA
]
ans.iloc[5]

question_id       6
answer_1       0.24
answer_2        NaN
answer_3        NaN
Name: 5, dtype: str

7. Рассмотрим простой линейный индекс риска орбитального витка:
Risk_Index=w1⋅battery_temp_mean40+w2⋅reaction_wheel_current_mean2.0+w3⋅attitude_error_mean1.5,
где веса w1, w2, w3 ≥ 0 и w1 + w2 + w3 = 1.
Предполагается, что отказ в ближайшие 12 часов (failure_in_12h = 1) происходит, когда индекс риска превышает порог t.
Необходимо подобрать значения w1, w2, w3 и порог t, максимизирующие F1-score классификации метки failure_in_12h. Правила расчёта: расчёт выполняется по всем виткам, без разбиения по спутникам; веса принимают значения на сетке с шагом 0.01 и w1 + w2 + w3 = 1; для каждого набора весов порог t подбирается перебором всех возможных порогов, то есть всех различных разбиений витков по значению индекса риска R: виток считается опасным, если R > t, иначе безопасным.
Каков максимальный F1-score, которого можно достичь при таком переборе? В ответе укажите только значение F1-score, округлённое до тысячных. 

In [43]:
x1 = df["battery_temp_mean"].to_numpy() / 40
x2 = df["reaction_wheel_current_mean"].to_numpy() / 2
x3 = df["attitude_error_mean"].to_numpy() / 1.5

f1_max = 0
target = df["failure_in_12h"].to_numpy()
p = target.sum()
counts = np.arange(1, n + 1)

for w1 in range(0, 100+1, 1):
    for w2 in range(0, 100+1-w1, 1):
        w3 = 100-w1-w2

        risk = w1*x1 + w2*x2 + w3*x3
        sort_idx = (-risk).argsort()
        target_sort = target[sort_idx]
        risk_sort = risk[sort_idx]

        tp = np.cumsum(target[sort_idx])
        f1 = 2*(tp)/(counts+p)

        valid = np.ones(len(target), dtype=bool)
        valid[:-1] = risk_sort[:-1] != risk_sort[1:]

        f1_max = max(f1_max, f1[valid].max())

ans.loc[6, ["answer_1", "answer_2", "answer_3"]] = [
    str(
        round(
            f1_max,
            3
        )
    ),
    pd.NA,
    pd.NA
]
ans.iloc[6]

question_id        7
answer_1       0.945
answer_2         NaN
answer_3         NaN
Name: 6, dtype: str

8. Известно, что проблемы с системой ориентации часто проявляются в виде нестабильности тока маховиков. Рассмотрим для каждого спутника временной ряд признака reaction_wheel_current_std (стандартное отклонение тока маховиков за виток).
Для каждого спутника выполните следующие расчёты:
Определите порог аномалии как
threshold = μ + 3 · σ,
где μ и σ - среднее и стандартное отклонение признака reaction_wheel_current_std по всем виткам данного спутника. Среднее и стандартное отклонение рассчитываются по всем виткам спутника с использованием ddof=0.
Найдите долю аномальных витков: долю витков, на которых reaction_wheel_current_std превышает порог threshold.
Рассчитайте коэффициент корреляции Пирсона между долей аномальных витков для спутника и общим количеством отказов (failure_in_12h) на этом спутнике.
Чему равен коэффициент корреляции Пирсона? Укажите ответ, округлённый до трёх знаков после запятой. 

In [44]:
g = df.groupby("satellite_id")
x = df["reaction_wheel_current_std"]

thr = (
    g["reaction_wheel_current_std"].transform("mean") 
    + 3*g["reaction_wheel_current_std"].transform(
        lambda s: s.std(ddof=0)
    ) 
)

stats = df.assign(is_anomaly=x>thr).groupby("satellite_id").agg(
    anomaly_rate=("is_anomaly", "mean"),
    failure_cnt=("failure_in_12h", "sum"),
)

res = stats["anomaly_rate"].corr(stats["failure_cnt"])

ans.loc[7, ["answer_1", "answer_2", "answer_3"]] = [
    str(
        round(
            res,
            3
        )
    ),
    pd.NA,
    pd.NA
]
ans.iloc[7]

question_id        8
answer_1       0.693
answer_2         NaN
answer_3         NaN
Name: 7, dtype: str

9. Постройте линейную регрессию для предсказания средней температуры батарей battery_temp_mean по двум признакам: attitude_error_mean и reaction_wheel_current_mean. Модель обучается на всех витках, чтобы оценить обычную линейную связь между этими признаками и температурой батарей.
Для каждого витка рассчитайте остаток как разность между фактической и предсказанной моделью температурой батарей: residual=battery_temp_mean−ˆbattery_temp_mean.
Затем вычислите стандартное отклонение остатков отдельно для двух групп витков: витки с failure_in_12h = 1 и витки с failure_in_12h = 0.
Чему равно отношение стандартного отклонения остатков для витков с failure_in_12h = 1 к стандартному отклонению остатков для витков с failure_in_12h = 0? Укажите ответ, округлённый до сотых. 

In [45]:
from sklearn.linear_model import LinearRegression

X = df[["attitude_error_mean", "reaction_wheel_current_mean"]]
y = df["battery_temp_mean"]

model = LinearRegression()
model.fit(X, y)
y_cap = model.predict(X)

residual = y-y_cap

residual_std0 = residual[df["failure_in_12h"] == 0].std()
residual_std1 = residual[df["failure_in_12h"] == 1].std()

res = residual_std1 / residual_std0

ans.loc[8, ["answer_1", "answer_2", "answer_3"]] = [
    str(
        round(
            res,
            2
        )
    ),
    pd.NA,
    pd.NA
]
ans.iloc[8]

question_id       9
answer_1       0.98
answer_2        NaN
answer_3        NaN
Name: 8, dtype: str

10. 
Оператор хочет построить весовой ансамбль из трёх моделей (из 10 доступных) для предиктивной диагностики отказов спутников. Ансамблевый прогноз риска для орбитального витка задаётся формулой:
Pensemble = w1 · pa + w2 · pb + w3 · pc,
где
a, b, c - номера выбранных моделей (различные целые числа от 1 до 10);
pa, pb, pc - прогнозы выбранных моделей для данного витка (столбцы model_a_pred, model_b_pred, model_c_pred);
веса w1, w2, w3 > 0 и w1 + w2 + w3 = 1.
Система использует пороговое правило:

    если Pensemble > t, виток считается опасным - назначается внеплановая проверка или манёвр;
    если Pensemble ≤ t, виток считается безопасным - полёт выполняется по плану.

Экономическая модель потерь:
- FN (пропущенный отказ: фактически failure_in_12h = 1, а ансамбль дал безопасный прогноз) - потери 20 000 000 руб.
- FP (ложная тревога: фактически failure_in_12h = 0, а ансамбль дал опасный прогноз) - потери 100 000 руб.
- TP и TN - потери 0 руб.
- Суммарные потери на всем датасете:
L = 20 000 000 · FN + 100 000 · FP.
На работу системы накладываются дополнительные ограничения:

    общая доля пропущенных отказов FNTP+FN≤10%;
    общая доля витков с предупреждениями TP+FPNtotal≤25%.

Спутники находятся на трёх типах орбит (orbit_type: 0 - LEO, 1 - MEO, 2 - GEO). Оператор требует, чтобы ограничения выполнялись отдельно для каждого типа орбиты:

    для каждой группы орбит: FNgroupTPgroup+FNgroup≤15%;
    для каждой группы орбит: TPgroup+FPgroupNgroup≤30%.

Таким образом, все ограничения должны выполняться одновременно: в целом по всем виткам и отдельно для LEO, MEO, GEO.

Требуется найти тройку моделей (a, b, c), веса (w1, w2, w3) и порог t, которые: удовлетворяют всем указанным ограничениям по долям пропусков и предупреждений; минимизируют суммарные потери L. При одинаковых потерях применяется единое правило выбора: сначала минимальный порог, затем лексикографически минимальная тройка моделей.
Найдите оптимальную комбинацию моделей, весов и порога. В качестве ответа укажите:

    Номера трёх моделей a, b, c в порядке возрастания.
    Порог t, округлённый до сотых.
    Минимальные суммарные потери L в рублях, округлённые до целого.


In [63]:
from itertools import combinations
import heapq

P = df[[f"model_{i}_pred" for i in range(1, 11)]].to_numpy()
target = df["failure_in_12h"].to_numpy()
orbit = df["orbit_type"].to_numpy()
n, p = len(target), target.sum()
p_g = np.array([target[orbit == g].sum() for g in range(3)])
n_g = np.array([(orbit == g).sum() for g in range(3)])
max_warnings = n // 4
counts = np.arange(1, max_warnings + 1)
checked = set()
best_by_triple = {}


def check(models, weights):
    if (models, weights) in checked:
        return
    checked.add((models, weights))

    a, b, c = models
    w1, w2, w3 = weights
    risk = w1 * P[:, a - 1] + w2 * P[:, b - 1] + w3 * P[:, c - 1]
    idx = np.argpartition(-risk, max_warnings)[:max_warnings + 1]
    idx = idx[np.argsort(-risk[idx])]
    risk_sorted = risk[idx]
    y_sorted = target[idx[:-1]]
    orbit_sorted = orbit[idx[:-1]]

    tp = np.cumsum(y_sorted)
    fn, fp = p - tp, counts - tp
    valid = (risk_sorted[:-1] != risk_sorted[1:]) & (fn / p <= 0.10)
    for g in range(3):
        mask = orbit_sorted == g
        fn_group = p_g[g] - np.cumsum(y_sorted * mask)
        warnings_group = np.cumsum(mask)
        valid &= fn_group / p_g[g] <= 0.15
        valid &= warnings_group / n_g[g] <= 0.30
    if not valid.any():
        return

    loss = np.where(valid, 200 * fn + fp, np.inf)
    k = np.flatnonzero(loss == loss.min())[-1]
    result = (loss[k], risk_sorted[k + 1] / 100, models, weights)
    if result < best_by_triple.get(models, (np.inf,)):
        best_by_triple[models] = result


for models in combinations(range(1, 11), 3):
    for w1 in range(20, 100, 20):
        for w2 in range(20, 100 - w1, 20):
            check(models, (w1, w2, 100 - w1 - w2))

best_key = min(best_by_triple.values())
positive = target == 1


def lower_bound(models, vertices):
    a, b, c = models
    risks = [w1 * P[:, a - 1] + w2 * P[:, b - 1] + w3 * P[:, c - 1]
             for w1, w2, w3 in vertices]
    low = np.minimum.reduce(risks)[~positive]
    high = np.maximum.reduce(risks)[positive]
    max_fn = int(best_key[0] // 200)
    thresholds = np.sort(np.partition(high, max_fn)[:max_fn + 1])
    return min(200 * k + (low > threshold + 1e-10).sum()
               for k, threshold in enumerate(thresholds))


def promising(bound, models):
    return bound < best_key[0] or (bound == best_key[0] and models != best_key[2])


queue = []
serial = 0
for models in combinations(range(1, 11), 3):
    vertices = np.eye(3) * 100
    bound = lower_bound(models, vertices)
    if promising(bound, models):
        heapq.heappush(queue, (bound, serial, models, vertices))
        serial += 1

while queue:
    bound, _, models, vertices = heapq.heappop(queue)
    if not promising(bound, models):
        continue
    check(models, tuple(vertices.mean(axis=0)))
    best_key = min(best_by_triple.values())
    i, j = max(((0, 1), (0, 2), (1, 2)),
               key=lambda pair: np.linalg.norm(vertices[pair[0]] - vertices[pair[1]]))
    midpoint = (vertices[i] + vertices[j]) / 2
    for replace in (i, j):
        child = vertices.copy()
        child[replace] = midpoint
        child_bound = lower_bound(models, child)
        if promising(child_bound, models):
            heapq.heappush(queue, (child_bound, serial, models, child))
            serial += 1

min_loss, t, abc, weights = best_key
best_weights = tuple(w / 100 for w in weights)


In [64]:
ans.loc[9, ["answer_1", "answer_2", "answer_3"]] = [
    ", ".join(map(str, abc)),
    str(round(t, 2)),
    str(round(min_loss*100_000))
]
ans.iloc[9]

question_id           10
answer_1        7, 9, 10
answer_2            0.37
answer_3       169800000
Name: 9, dtype: str

In [65]:
ans.to_csv("ans.csv", index=False)